In [31]:
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error, mean_squared_error
import optuna
from optuna.samplers import TPESampler
import warnings
warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

In [32]:
# ==================== 时序块交叉验证 ====================

class TimeSeriesBlockCV:
    """
    时序块交叉验证
    验证集为连续块，训练集为剩余部分（最多两块）
    
    例如 n=10000, cv_folds=5:
    Fold 1: val=[0:2000],     train=[2000:10000]
    Fold 2: val=[2000:4000],  train=[0:2000] + [4000:10000]
    Fold 3: val=[4000:6000],  train=[0:4000] + [6000:10000]
    Fold 4: val=[6000:8000],  train=[0:6000] + [8000:10000]
    Fold 5: val=[8000:10000], train=[0:8000]
    """
    
    def __init__(self, n_splits=5):
        self.n_splits = n_splits
    
    def split(self, X):
        n_samples = len(X)
        fold_size = n_samples // self.n_splits
        
        for i in range(self.n_splits):
            val_start = i * fold_size
            val_end = (i + 1) * fold_size if i < self.n_splits - 1 else n_samples
            
            val_idx = np.arange(val_start, val_end)
            train_idx = np.concatenate([
                np.arange(0, val_start),
                np.arange(val_end, n_samples)
            ])
            
            yield train_idx, val_idx
    
    def get_n_splits(self):
        return self.n_splits

In [33]:
# ==================== Optuna优化版本 ====================

def xgboost_optuna_cv(X_train, y_train, X_test, y_test, cat_indices=None,
                       n_trials=30, cv_folds=5, random_state=42, use_gpu=False):
    """
    XGBoost Optuna + 时序块交叉验证 + 测试集评估
    
    参数:
    ------
    X_train, y_train : 训练数据
    X_test, y_test : 测试数据
    cat_indices : list
        类别特征索引，如 [8, 9, 10, 11, 12, 13, 14]
    n_trials : int
        优化迭代次数
    cv_folds : int
        交叉验证折数
    use_gpu : bool
        是否使用GPU
    """
    
    # 确保是numpy数组
    y_train = np.array(y_train)
    y_test = np.array(y_test)
    
    # 转为DataFrame并设置类别特征
    X_train_df = pd.DataFrame(X_train)
    X_test_df = pd.DataFrame(X_test)
    if cat_indices:
        for col in cat_indices:
            X_train_df[col] = X_train_df[col].astype('category')
            X_test_df[col] = X_test_df[col].astype('category')
    
    print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
    print(f"类别特征: {cat_indices}")
    print(f"优化次数: {n_trials}, 时序CV: {cv_folds}折, GPU: {use_gpu}")
    print("=" * 60)
    
    # 时序块交叉验证
    tscv = TimeSeriesBlockCV(n_splits=cv_folds)
    
    # 打印CV划分信息
    print("时序块划分:")
    for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_df)):
        print(f"  Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)} "
              f"[val范围: {val_idx[0]}-{val_idx[-1]}]")
    print("=" * 60)
    
    all_results = []
    
    def objective(trial):
        params = {
            'n_estimators': trial.suggest_int('n_estimators', 200, 2000),
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 1.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 1.0, log=True),
            'gamma': trial.suggest_float('gamma', 1e-8, 1.0, log=True),
        }
        
        fold_metrics = {'MAE': [], 'RMSE': [], 'MAPE': []}
        
        for train_idx, val_idx in tscv.split(X_train_df):
            X_tr = X_train_df.iloc[train_idx]
            X_val = X_train_df.iloc[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
            dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
            dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
            
            xgb_params = {
                'objective': 'reg:squarederror',
                'eval_metric': 'mae',
                'tree_method': 'gpu_hist' if use_gpu else 'hist',
                'device': 'cuda' if use_gpu else 'cpu',
                'seed': random_state,
                'verbosity': 0,
                **{k: v for k, v in params.items() if k != 'n_estimators'}
            }
            
            model = xgb.train(
                xgb_params, dtrain,
                num_boost_round=params['n_estimators'],
                evals=[(dval, 'val')],
                early_stopping_rounds=30,
                verbose_eval=False
            )
            
            y_pred = model.predict(dval)
            fold_metrics['MAE'].append(mean_absolute_error(y_val, y_pred))
            fold_metrics['RMSE'].append(rmse(y_val, y_pred))
            fold_metrics['MAPE'].append(mape(y_val, y_pred))
        
        result = {
            'trial': trial.number + 1,
            'params': params,
            'MAE_mean': np.mean(fold_metrics['MAE']),
            'MAE_std': np.std(fold_metrics['MAE']),
            'RMSE_mean': np.mean(fold_metrics['RMSE']),
            'RMSE_std': np.std(fold_metrics['RMSE']),
            'MAPE_mean': np.mean(fold_metrics['MAPE']),
            'MAPE_std': np.std(fold_metrics['MAPE']),
        }
        all_results.append(result)
        
        print(f"Trial {trial.number+1:3d}: MAE={result['MAE_mean']:.4f}, "
              f"RMSE={result['RMSE_mean']:.4f}, MAPE={result['MAPE_mean']:.2f}%")
        
        return result['MAE_mean']
    
    # 优化
    study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=random_state))
    study.optimize(objective, n_trials=n_trials, show_progress_bar=True)
    
    best_params = study.best_params
    
    # 最终模型
    print("\n" + "=" * 60)
    print("使用最佳参数训练最终模型...")
    
    dtrain_full = xgb.DMatrix(X_train_df, label=y_train, enable_categorical=True)
    dtest = xgb.DMatrix(X_test_df, label=y_test, enable_categorical=True)
    
    final_params = {
        'objective': 'reg:squarederror',
        'tree_method': 'gpu_hist' if use_gpu else 'hist',
        'device': 'cuda' if use_gpu else 'cpu',
        'seed': random_state,
        'verbosity': 0,
        **{k: v for k, v in best_params.items() if k != 'n_estimators'}
    }
    best_model = xgb.train(final_params, dtrain_full, num_boost_round=best_params['n_estimators'])
    
    # 输出
    print("\n" + "=" * 60)
    print("最佳参数:")
    for k, v in best_params.items():
        print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")
    
    best_cv_result = min(all_results, key=lambda x: x['MAE_mean'])
    print(f"\n【交叉验证性能】")
    print(f"  MAE:  {best_cv_result['MAE_mean']:.4f} ± {best_cv_result['MAE_std']:.4f}")
    print(f"  RMSE: {best_cv_result['RMSE_mean']:.4f} ± {best_cv_result['RMSE_std']:.4f}")
    print(f"  MAPE: {best_cv_result['MAPE_mean']:.2f}% ± {best_cv_result['MAPE_std']:.2f}%")
    
    # 测试集评估
    print(f"\n【测试集性能】")
    y_pred_test = best_model.predict(dtest)
    test_metrics = evaluate(y_test, y_pred_test, prefix="  ")
    
    return best_model, best_params, all_results, test_metrics, y_pred_test

In [34]:
# ==================== 保存模型 ====================

def save_model(model, best_params, test_metrics, model_name, save_dir='./models'):
    """
    保存模型、参数和指标
    
    参数:
        model: 训练好的模型
        best_params: 最佳参数字典
        test_metrics: 测试指标字典
        model_name: 模型名称 ('xgboost', 'lightgbm', 'catboost')
        save_dir: 保存目录
    """
    import os
    os.makedirs(save_dir, exist_ok=True)
    
    # 保存模型
    if model_name == 'xgboost':
        model.save_model(f'{save_dir}/{model_name}_model.json')
    elif model_name == 'lightgbm':
        model.save_model(f'{save_dir}/{model_name}_model.txt')
    elif model_name == 'catboost':
        model.save_model(f'{save_dir}/{model_name}_model.cbm')
    
    # 保存参数和指标
    info = {'best_params': best_params, 'test_metrics': test_metrics}
    with open(f'{save_dir}/{model_name}_info.json', 'w') as f:
        json.dump(info, f, indent=2)
    
    print(f"模型已保存到 {save_dir}/{model_name}_model.*")

In [35]:
# import time
# import multiprocessing as mp
# from multiprocessing import Pool as ProcessPool, Manager

# def process_single_region_with_progress(args):
#     """处理单个区域的模型训练（带进度共享）"""
#     name, cat_indices, n_trials, cv_folds, use_gpu, progress_dict = args
    
#     try:
#         progress_dict[name] = "加载数据..."
        
#         # 加载数据
#         X_2d = np.load(f'./train_data/TD_X_2d_{name}.npy')
#         y = np.load(f'./train_data/TD_y_{name}.npy')
        
#         progress_dict[name] = "划分数据集..."
        
#         # 划分数据集
#         test_size = 24 * 365 * 2
#         X_train, X_test = X_2d[:-test_size], X_2d[-test_size:]
#         y_train, y_test = y[:-test_size], y[-test_size:]
        
#         progress_dict[name] = "训练模型..."
        
#         # 训练模型
#         best_model, best_params, cv_results, test_metrics, y_pred = xgboost_optuna_cv(
#             X_train, y_train, X_test, y_test,
#             cat_indices=cat_indices,
#             n_trials=n_trials,
#             cv_folds=cv_folds,
#             use_gpu=use_gpu
#         )
        
#         progress_dict[name] = "保存模型..."
        
#         # 保存模型
#         save_model(best_model, best_params, test_metrics, name, "xgboost")
        
#         progress_dict[name] = "完成"
#         return (True, name, test_metrics)
    
#     except Exception as e:
#         progress_dict[name] = f"错误: {str(e)}"
#         return (False, name, str(e))


# def monitor_progress(progress_dict, names, interval=5):
#     """监控进度的辅助函数"""
#     while True:
#         print("\n--- 当前进度 ---")
#         completed = 0
#         for name in names:
#             status = progress_dict.get(name, "等待中...")
#             print(f"  {name}: {status}")
#             if status == "完成" or status.startswith("错误"):
#                 completed += 1
        
#         if completed == len(names):
#             break
#         time.sleep(interval)


# def run_multiprocess_training_v2(names, cat_indices, n_trials=30, cv_folds=5, use_gpu=False):
#     """多进程运行训练（增强版）"""
    
#     num_processes = min(len(names), mp.cpu_count())
#     print(f"使用 {num_processes} 个进程进行训练...")
#     print(f"待处理区域: {names}")
    
#     # 使用 Manager 创建共享字典
#     manager = Manager()
#     progress_dict = manager.dict()
    
#     # 准备参数
#     args_list = [
#         (name, cat_indices, n_trials, cv_folds, use_gpu, progress_dict) 
#         for name in names
#     ]
    
#     # 启动进程池
#     with ProcessPool(processes=num_processes) as pool:
#         # 异步执行
#         async_result = pool.map_async(process_single_region_with_progress, args_list)
        
#         # 等待完成，同时可以监控进度
#         while not async_result.ready():
#             # 打印当前状态
#             status_line = " | ".join([f"{n}:{progress_dict.get(n, '等待')[:6]}" for n in names])
#             print(f"\r{status_line}", end="", flush=True)
#             time.sleep(2)
        
#         results = async_result.get()
    
#     print()  # 换行
#     return results


# if __name__ == '__main__':
#     names = ['COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST', 'ERCOT']
    
#     results = run_multiprocess_training_v2(
#         names=names,
#         cat_indices=[8, 9, 10, 11, 12, 13, 14],
#         n_trials=30,
#         cv_folds=5,
#         use_gpu=True
#     )
    
#     # 结果汇总
#     print("\n" + "="*60)
#     print("训练结果汇总:")
#     print("="*60)
    
#     for success, name, info in results:
#         status = "✓ 成功" if success else "✗ 失败"
#         print(f"{status} | {name:8s} | {info}")

In [ ]:
# names = ['Date', 'COAST', 'EAST', 'FWEST', 'NORTH', 'NCENT', 'SOUTH', 'SCENT', 'WEST', 'ERCOT']
# for i, v in enumerate()
#     X_2d = np.load('./train_data/train_data_X_2d.npy')
#     y = np.load('./train_data/train_data_y.npy')

#     print("划分数据集...")
#     test_size = 24 * 365 * 2  # 最后2年作为测试集

#     X_train, X_test = X_2d[:-test_size], X_2d[-test_size:]
#     y_train, y_test = y[:-test_size], y[-test_size:]

In [41]:
X_2d = np.load('./train_data/train_data_X_2d.npy')
y = np.load('./train_data/train_data_y.npy')

print("划分数据集...")
test_size = 24 * 365 * 1  # 最后2年作为测试集

X_train, X_test = X_2d[:-test_size], X_2d[-test_size:]
y_train, y_test = y[:-test_size], y[-test_size:]

划分数据集...


In [43]:
best_model, best_params, all_results, test_metrics, y_pred_test = xgboost_optuna_cv(
    X_train, y_train,
    X_test, y_test,
    cat_indices=[8, 9, 10, 11, 12, 13, 14],
    n_trials=30,
    cv_folds=5,
    use_gpu=True
)

训练集: (69912, 21), 测试集: (8760, 21)
类别特征: [8, 9, 10, 11, 12, 13, 14]
优化次数: 30, 时序CV: 5折, GPU: True
时序块划分:
  Fold 1: train=55930, val=13982 [val范围: 0-13981]
  Fold 2: train=55930, val=13982 [val范围: 13982-27963]
  Fold 3: train=55930, val=13982 [val范围: 27964-41945]
  Fold 4: train=55930, val=13982 [val范围: 41946-55927]
  Fold 5: train=55928, val=13984 [val范围: 55928-69911]


  0%|          | 0/30 [00:00<?, ?it/s]

Trial   1: MAE=2282.5525, RMSE=3223.0645, MAPE=5.15%
Trial   2: MAE=2158.9605, RMSE=3062.5091, MAPE=4.87%
Trial   3: MAE=2164.5376, RMSE=3085.2121, MAPE=4.88%
Trial   4: MAE=2179.5542, RMSE=3095.0697, MAPE=4.91%
Trial   5: MAE=2148.4536, RMSE=3052.1778, MAPE=4.84%
Trial   6: MAE=2159.5448, RMSE=3077.8975, MAPE=4.87%
Trial   7: MAE=2230.1037, RMSE=3172.3567, MAPE=5.02%
Trial   8: MAE=2167.7231, RMSE=3083.0103, MAPE=4.88%
Trial   9: MAE=2247.2283, RMSE=3183.7229, MAPE=5.07%
Trial  10: MAE=2153.9242, RMSE=3067.6478, MAPE=4.86%
Trial  11: MAE=2144.1801, RMSE=3044.8685, MAPE=4.84%
Trial  12: MAE=2150.3133, RMSE=3050.5330, MAPE=4.85%
Trial  13: MAE=2160.7925, RMSE=3061.2583, MAPE=4.88%
Trial  14: MAE=2167.3211, RMSE=3100.8034, MAPE=4.88%
Trial  15: MAE=2164.7037, RMSE=3079.1684, MAPE=4.88%
Trial  16: MAE=2156.5955, RMSE=3063.6436, MAPE=4.87%
Trial  17: MAE=2237.2341, RMSE=3188.5683, MAPE=5.04%
Trial  18: MAE=2164.5211, RMSE=3080.4964, MAPE=4.88%
Trial  19: MAE=2150.1369, RMSE=3059.9988, MAPE

In [3]:
# # ==================== 快速网格搜索版本 ====================

# def xgboost_fast_cv(X_train, y_train, X_test, y_test, cat_indices=None,
#                     cv_folds=5, random_state=42, use_gpu=False):
#     """快速网格搜索 + 时序块交叉验证"""
#     from itertools import product
    
#     y_train = np.array(y_train)
#     y_test = np.array(y_test)
    
#     X_train_df = pd.DataFrame(X_train)
#     X_test_df = pd.DataFrame(X_test)
#     if cat_indices:
#         for col in cat_indices:
#             X_train_df[col] = X_train_df[col].astype('category')
#             X_test_df[col] = X_test_df[col].astype('category')
    
#     param_grid = {
#         'n_estimators': [200, 400, 600, 800, 1000],
#         'max_depth': [3, 4, 5],
#         'learning_rate': [0.05],
#         'min_child_weight': [1, 3, 5],
#     }
    
#     param_names = list(param_grid.keys())
#     param_combinations = list(product(*param_grid.values()))
#     total = len(param_combinations)
    
#     print(f"训练集: {X_train.shape}, 测试集: {X_test.shape}")
#     print(f"参数组合: {total}, 时序CV: {cv_folds}折")
#     print("=" * 60)
    
#     tscv = TimeSeriesBlockCV(n_splits=cv_folds)
    
#     # 打印划分
#     print("时序块划分:")
#     for fold, (train_idx, val_idx) in enumerate(tscv.split(X_train_df)):
#         print(f"  Fold {fold+1}: train={len(train_idx)}, val={len(val_idx)} "
#               f"[val范围: {val_idx[0]}-{val_idx[-1]}]")
#     print("=" * 60)
    
#     all_results = []
#     best_mae = float('inf')
#     best_params = None
    
#     for idx, params in enumerate(param_combinations):
#         current_params = dict(zip(param_names, params))
#         fold_metrics = {'MAE': [], 'RMSE': [], 'MAPE': []}
        
#         for train_idx, val_idx in tscv.split(X_train_df):
#             X_tr = X_train_df.iloc[train_idx]
#             X_val = X_train_df.iloc[val_idx]
#             y_tr, y_val = y_train[train_idx], y_train[val_idx]
            
#             dtrain = xgb.DMatrix(X_tr, label=y_tr, enable_categorical=True)
#             dval = xgb.DMatrix(X_val, label=y_val, enable_categorical=True)
            
#             xgb_params = {
#                 'objective': 'reg:squarederror',
#                 'tree_method': 'gpu_hist' if use_gpu else 'hist',
#                 'device': 'cuda' if use_gpu else 'cpu',
#                 'verbosity': 0,
#                 **{k: v for k, v in current_params.items() if k != 'n_estimators'}
#             }
            
#             model = xgb.train(
#                 xgb_params, dtrain,
#                 num_boost_round=current_params['n_estimators'],
#                 evals=[(dval, 'val')],
#                 early_stopping_rounds=50,
#                 verbose_eval=False
#             )
            
#             y_pred = model.predict(dval)
#             fold_metrics['MAE'].append(mean_absolute_error(y_val, y_pred))
#             fold_metrics['RMSE'].append(rmse(y_val, y_pred))
#             fold_metrics['MAPE'].append(mape(y_val, y_pred))
        
#         result = {
#             'params': current_params,
#             'MAE_mean': np.mean(fold_metrics['MAE']),
#             'MAE_std': np.std(fold_metrics['MAE']),
#             'RMSE_mean': np.mean(fold_metrics['RMSE']),
#             'RMSE_std': np.std(fold_metrics['RMSE']),
#             'MAPE_mean': np.mean(fold_metrics['MAPE']),
#             'MAPE_std': np.std(fold_metrics['MAPE']),
#         }
#         all_results.append(result)
        
#         print(f"[{idx+1}/{total}] {current_params} → MAE={result['MAE_mean']:.4f}")
        
#         if result['MAE_mean'] < best_mae:
#             best_mae = result['MAE_mean']
#             best_params = current_params
    
#     # 最终模型
#     print("\n" + "=" * 60)
#     dtrain_full = xgb.DMatrix(X_train_df, label=y_train, enable_categorical=True)
#     dtest = xgb.DMatrix(X_test_df, label=y_test, enable_categorical=True)
    
#     final_params = {
#         'objective': 'reg:squarederror',
#         'tree_method': 'gpu_hist' if use_gpu else 'hist',
#         'device': 'cuda' if use_gpu else 'cpu',
#         'verbosity': 0,
#         **{k: v for k, v in best_params.items() if k != 'n_estimators'}
#     }
#     best_model = xgb.train(final_params, dtrain_full, num_boost_round=best_params['n_estimators'])
    
#     print(f"最佳参数: {best_params}")
    
#     best_cv_result = min(all_results, key=lambda x: x['MAE_mean'])
#     print(f"\n【交叉验证性能】")
#     print(f"  MAE:  {best_cv_result['MAE_mean']:.4f} ± {best_cv_result['MAE_std']:.4f}")
#     print(f"  RMSE: {best_cv_result['RMSE_mean']:.4f} ± {best_cv_result['RMSE_std']:.4f}")
#     print(f"  MAPE: {best_cv_result['MAPE_mean']:.2f}% ± {best_cv_result['MAPE_std']:.2f}%")
    
#     # 测试集
#     print(f"\n【测试集性能】")
#     y_pred_test = best_model.predict(dtest)
#     test_metrics = evaluate(y_test, y_pred_test, prefix="  ")
    
#     return best_model, best_params, all_results, test_metrics, y_pred_test

In [4]:
# best_model, best_params, all_results, test_metrics, y_pred_test = xgboost_fast_cv(
#     X_train, y_train, X_test, y_test,
#     cat_indices=[8, 9, 10, 11, 12, 13, 14],
#     cv_folds=5,
#     use_gpu=True
# )

In [50]:
# ==================== 读取模型 ====================

def load_model(model_name, save_dir='./models'):
    """
    读取模型、参数和指标
    
    返回:
        model, best_params, test_metrics
    """
    import xgboost as xgb
    import lightgbm as lgb
    from catboost import CatBoostRegressor
    
    # 读取模型
    if model_name == 'xgboost':
        model = xgb.Booster()
        model.load_model(f'{save_dir}/{model_name}_model.json')
    elif model_name == 'lightgbm':
        model = lgb.Booster(model_file=f'{save_dir}/{model_name}_model.txt')
    elif model_name == 'catboost':
        model = CatBoostRegressor()
        model.load_model(f'{save_dir}/{model_name}_model.cbm')
    
    # 读取参数和指标
    with open(f'{save_dir}/{model_name}_info.json', 'r') as f:
        info = json.load(f)
    
    print(f"模型已从 {save_dir}/{model_name}_model.* 加载")
    return model, info['best_params'], info['test_metrics']

In [44]:
save_model(best_model, best_params, test_metrics, "xgboost")

模型已保存到 ./models/xgboost_model.*


In [51]:
model, params, metrics = load_model('xgboost')

模型已从 ./models/xgboost_model.* 加载
